In [1]:
import copy
from tqdm import tqdm
from torch.autograd import Variable
import argparse
import time
import os
from src.language_models.dictionary_corpus import Dictionary, tokenize
import pandas
import torch
import numpy as np 
import h5py


In [2]:
#constructing test set with the 1000 sentences test
nounpp = '/scratch2/mrenaudin/colorlessgreenRNNs/nounpp.txt'

In [ ]:
class NounPPDataset(Dataset):
    def __init__(self, nounpp_file, dictionary):
        self.sentences = []
        self.verbs = []
        self.conditions = []
        self.correctness = []
        self.ids = []
        self.encoded_sentences=[]
        self.encoded_verbs = []
        self.dictionary = dictionary

        with open(nounpp_file, "r") as f:
            for line in f:
                line = line.split()
                #sentence = line[:6]  
                sentence = line[:1]
                #verb = line[5]
                verb = line[2]       
                #condition = line[6]  
                condition = line[3]
                #correctness = line[7]
                correctness = line[4]
                #id = int(line[8][2:]) 
                id = int(line[5][2:]) 
                encoded_sentence = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence]
                encoded_verb = self.dictionary.word2idx.get(verb, self.dictionary.word2idx.get("<unk>"))

                self.sentences.append(sentence)
                self.verbs.append(verb)
                self.conditions.append(condition)
                self.correctness.append(correctness)
                self.ids.append(id)
                self.encoded_sentences.append(encoded_sentence)
                self.encoded_verbs.append(encoded_verb)

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return {
            "sentence": self.sentences[idx],
            "encoded_sentence": torch.tensor(self.encoded_sentences[idx], dtype=torch.long),
            "verb": self.verbs[idx],
            "encoded_verb": torch.tensor(self.encoded_verbs[idx], dtype=torch.long),
            "condition": self.conditions[idx],
            "correctness":self.correctness[idx],
            "id": torch.tensor(self.ids[idx], dtype=torch.long),
        }
